In [7]:
import os
from pathlib import Path
from typing import Tuple, List, Dict

from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms


class ControlsFolderDataset(Dataset):
    """
    Dataset that reads images from:
        root/
          Break/
          Left/
          Mirror/
          None/
          Right/
    and uses the folder names as class labels.

    - Resizes to `image_size` (H, W)
    - Converts to float tensor in [0, 1] by dividing by 255 (via ToTensor)
    """

    IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

    def __init__(
        self,
        root_dir: str,
        image_size: Tuple[int, int] = (96, 96),
        class_names: List[str] | None = None,
    ):
        self.root_dir = Path(root_dir)
        if not self.root_dir.exists():
            raise FileNotFoundError(f"Root directory not found: {self.root_dir}")

        # Discover class folders
        if class_names is None:
            class_folders = [p for p in self.root_dir.iterdir() if p.is_dir()]
            if not class_folders:
                raise ValueError(f"No class subfolders found in: {self.root_dir}")
            self.class_names = sorted([p.name for p in class_folders])
        else:
            self.class_names = list(class_names)

        self.class_to_idx: Dict[str, int] = {c: i for i, c in enumerate(self.class_names)}

        # Collect samples: (image_path, label_idx)
        self.samples: List[Tuple[Path, int]] = []
        for cname in self.class_names:
            cdir = self.root_dir / cname
            if not cdir.exists():
                raise FileNotFoundError(f"Class folder missing: {cdir}")

            for f in cdir.rglob("*"):
                if f.is_file() and f.suffix.lower() in self.IMG_EXTS:
                    self.samples.append((f, self.class_to_idx[cname]))

        if not self.samples:
            raise ValueError(
                f"No images found under {self.root_dir}. "
                f"Expected image extensions: {sorted(self.IMG_EXTS)}"
            )

        # Transform: resize -> tensor([0,1]) (div by 255) -> (optional) ensure 3ch
        self.transform = transforms.Compose([
            transforms.Resize(image_size, interpolation=transforms.InterpolationMode.BILINEAR),
            transforms.ToTensor(),  # converts uint8 [0..255] to float32 [0..1] (div by 255)
        ])

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx: int):
        img_path, label = self.samples[idx]

        # Open with PIL and ensure RGB
        with Image.open(img_path) as im:
            im = im.convert("RGB")
            x = self.transform(im)

        y = torch.tensor(label, dtype=torch.long)
        return x, y, str(img_path)


if __name__ == "__main__":
    root = r"D:\Data\Game Dataset\Controls"
    image_size = (224, 224)

    ds = ControlsFolderDataset(root_dir=root, image_size=image_size)
    print("Classes:", ds.class_names)
    print("Num samples:", len(ds))

    dl = DataLoader(ds, batch_size=32, shuffle=True)

    xb, yb, paths = next(iter(dl))
    print("Batch x:", xb.shape, xb.dtype, xb.min().item(), xb.max().item())
    print("Batch y:", yb.shape, yb.dtype)
    print("Example path:", paths[0])


Classes: ['Left', 'None', 'Right']
Num samples: 7931
Batch x: torch.Size([32, 3, 224, 224]) torch.float32 0.0 1.0
Batch y: torch.Size([32]) torch.int64
Example path: D:\Data\Game Dataset\Controls\None\frame_056765_t_000967368935us.jpg


In [2]:
import torch
import torch.nn as nn


class ControlsCNN(nn.Module):
    """
    Simple CNN classifier for control images.
    Input:  (B, 3, H, W)  e.g. (B, 3, 224, 224)
    Output: (B, num_classes) logits
    """
    def __init__(self, num_classes: int = 3, in_channels: int = 3, dropout: float = 0.3):
        super().__init__()

        self.features = nn.Sequential(
            # Block 1
            nn.Conv2d(in_channels, 32, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2),  # /2

            # Block 2
            nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2),  # /4

            # Block 3
            nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2),  # /8

            # Block 4
            nn.Conv2d(128, 256, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2),  # /16

            # Compress spatial dims regardless of input size
            nn.AdaptiveAvgPool2d((1, 1)),  # -> (B, 256, 1, 1)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),                 # -> (B, 256)
            nn.Dropout(p=dropout),
            nn.Linear(256, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(p=dropout),
            nn.Linear(128, num_classes),  # logits
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.features(x)
        x = self.classifier(x)
        return x


In [3]:
# Instantiate based on your dataset classes
num_classes = 3  # Left, None, Right
model = ControlsCNN(num_classes=num_classes)

# Quick shape sanity check
xb = torch.randn(4, 3, 224, 224)
logits = model(xb)
print("logits shape:", logits.shape)  # expected: (4, 3)

logits shape: torch.Size([4, 3])


In [24]:
import torch
import torch.nn as nn
from torchvision.models import vit_b_16, ViT_B_16_Weights


class ControlsViT(nn.Module):
    def __init__(self, num_classes: int = 3, freeze_backbone: bool = True):
        super().__init__()

        # Load pretrained ViT
        weights = ViT_B_16_Weights.IMAGENET1K_V1
        self.backbone = vit_b_16(weights=weights)

        # Replace classification head
        in_features = self.backbone.heads.head.in_features
        self.backbone.heads.head = nn.Linear(in_features, num_classes)

        # Optionally freeze transformer backbone
        if freeze_backbone:
            for name, param in self.backbone.named_parameters():
                if not name.startswith("heads"):
                    param.requires_grad = False

    def forward(self, x):
        return self.backbone(x)


In [25]:
num_classes = len(ds.class_names)  # ['Left', 'None', 'Right']

model = ControlsViT(
    num_classes=num_classes,
    freeze_backbone=True  # very important with only ~70 samples
)

model


Downloading: "https://download.pytorch.org/models/vit_b_16-c867db91.pth" to C:\Users\emmanuelasare/.cache\torch\hub\checkpoints\vit_b_16-c867db91.pth
100%|██████████| 330M/330M [00:03<00:00, 99.9MB/s] 


ControlsViT(
  (backbone): VisionTransformer(
    (conv_proj): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
    (encoder): Encoder(
      (dropout): Dropout(p=0.0, inplace=False)
      (layers): Sequential(
        (encoder_layer_0): EncoderBlock(
          (ln_1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
          (self_attention): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=768, out_features=768, bias=True)
          )
          (dropout): Dropout(p=0.0, inplace=False)
          (ln_2): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
          (mlp): MLPBlock(
            (0): Linear(in_features=768, out_features=3072, bias=True)
            (1): GELU(approximate='none')
            (2): Dropout(p=0.0, inplace=False)
            (3): Linear(in_features=3072, out_features=768, bias=True)
            (4): Dropout(p=0.0, inplace=False)
          )
        )
        (encoder_layer_1): EncoderBlock(
          (ln_1): 

In [29]:
from torchvision import transforms
from torchvision.models import ViT_B_16_Weights

from torchvision.models import ViT_B_16_Weights

vit_weights = ViT_B_16_Weights.IMAGENET1K_V1
train_transform = vit_weights.transforms()



In [28]:
vit_weights

ViT_B_16_Weights.IMAGENET1K_V1

In [13]:
import time
import torch
import torch.nn as nn


def train_model(
    model: torch.nn.Module,
    dataloader,
    epochs: int = 15,
    lr: float = 1e-3,
    weight_decay: float = 1e-4,
    device: str | None = None,
    save_path: str = "controls_cnn.pt",
    print_every: int = 20,
):
    """
    Train a classification model using only training data (no validation).

    - model outputs logits (B, num_classes)
    - targets are int64 class indices
    """

    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"

    model = model.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    criterion = nn.CrossEntropyLoss()

    # Optional: mixed precision for speed on CUDA
    use_amp = (device == "cuda")
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

    num_samples = len(dataloader.dataset)
    # Save final model checkpoint
    ckpt = {
        "model_state_dict": model.state_dict(),
        "num_classes": getattr(model, "num_classes", None),
    }
    for epoch in range(1, epochs + 1):
        model.train()

        epoch_loss = 0.0
        epoch_correct = 0
        seen = 0

        t0 = time.time()

        for step, batch in enumerate(dataloader, start=1):
            # Your dataset returns (x, y, path)
            if len(batch) == 3:
                x, y, _paths = batch
            else:
                x, y = batch

            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            with torch.cuda.amp.autocast(enabled=use_amp):
                logits = model(x)
                loss = criterion(logits, y)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            # Stats
            batch_size = y.size(0)
            epoch_loss += loss.item() * batch_size
            preds = logits.argmax(dim=1)
            epoch_correct += (preds == y).sum().item()
            seen += batch_size

            if (step % print_every == 0) or (step == 1) or (step == len(dataloader)):
                avg_loss = epoch_loss / max(seen, 1)
                avg_acc = epoch_correct / max(seen, 1)
                print(
                    f"Epoch {epoch:02d}/{epochs} | "
                    f"Step {step:04d}/{len(dataloader):04d} | "
                    f"loss={avg_loss:.4f} acc={avg_acc:.4f}"
                )

        epoch_time = time.time() - t0
        epoch_avg_loss = epoch_loss / max(seen, 1)
        epoch_avg_acc = epoch_correct / max(seen, 1)

        print(
            f"Epoch {epoch:02d} done | "
            f"loss={epoch_avg_loss:.4f} acc={epoch_avg_acc:.4f} | "
            f"seen={seen}/{num_samples} | "
            f"time={epoch_time:.1f}s"
        )
        torch.save(ckpt, save_path)
        print(f"Saved model to: {save_path}")

    torch.save(ckpt, save_path)
    print(f"Saved model to: {save_path}")

    return model


In [8]:
from torch.utils.data import DataLoader

root = r"D:\Data\Game Dataset\Controls"
ds = ControlsFolderDataset(root_dir=root, image_size=(224, 224))

dl = DataLoader(ds, batch_size=32, shuffle=True)

In [14]:
trained_model = train_model(model, dl, epochs=1000, lr=1e-3, save_path="controls_cnn.pt")

C:\Users\emmanuelasare\AppData\Local\Temp\ipykernel_22272\1432040263.py:32: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
C:\Users\emmanuelasare\AppData\Local\Temp\ipykernel_22272\1432040263.py:61: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp):


Epoch 01/1000 | Step 0001/0248 | loss=0.8001 acc=0.6562
Epoch 01/1000 | Step 0020/0248 | loss=0.7605 acc=0.6422
Epoch 01/1000 | Step 0040/0248 | loss=0.7548 acc=0.6602
Epoch 01/1000 | Step 0060/0248 | loss=0.7321 acc=0.6625
Epoch 01/1000 | Step 0080/0248 | loss=0.7095 acc=0.6742
Epoch 01/1000 | Step 0100/0248 | loss=0.6964 acc=0.6791
Epoch 01/1000 | Step 0120/0248 | loss=0.6752 acc=0.6909
Epoch 01/1000 | Step 0140/0248 | loss=0.6564 acc=0.7018
Epoch 01/1000 | Step 0160/0248 | loss=0.6379 acc=0.7131
Epoch 01/1000 | Step 0180/0248 | loss=0.6146 acc=0.7269
Epoch 01/1000 | Step 0200/0248 | loss=0.5963 acc=0.7362
Epoch 01/1000 | Step 0220/0248 | loss=0.5806 acc=0.7440
Epoch 01/1000 | Step 0240/0248 | loss=0.5623 acc=0.7534
Epoch 01/1000 | Step 0248/0248 | loss=0.5563 acc=0.7555
Epoch 01 done | loss=0.5563 acc=0.7555 | seen=7931/7931 | time=58.7s
Saved model to: controls_cnn.pt
Epoch 02/1000 | Step 0001/0248 | loss=0.2992 acc=0.9062
Epoch 02/1000 | Step 0020/0248 | loss=0.4540 acc=0.8187
Epo

KeyboardInterrupt: 

In [22]:
model = ControlsCNN(num_classes=len(ds.class_names))

In [4]:
#load the saved weights
ckpt_path = "controls_cnn.pt"
ckpt = torch.load(ckpt_path, map_location="cpu")
model.load_state_dict(ckpt["model_state_dict"])

C:\Users\emmanuelasare\AppData\Local\Temp\ipykernel_22772\2977444609.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(ckpt_path, map_location="cpu")


<All keys matched successfully>

In [ ]:
from torch.utils.data import DataLoader

root = r"D:\Data\Game Dataset\Controls"
ds = ControlsFolderDataset(root_dir=root, image_size=(224, 224))
ds.transform = train_transform  # apply vit transform
dl = DataLoader(ds, batch_size=32, shuffle=True)

model = ControlsCNN(num_classes=len(ds.class_names))
trained_model = train_model(model, dl, epochs=1000, lr=1e-3, save_path="controls_cnn.pt")

In [38]:
from torch.utils.data import DataLoader

root = r"D:\Data\Game Dataset\Controls"
ds = ControlsFolderDataset(root_dir=root, image_size=(224, 224))

dl = DataLoader(ds, batch_size=32, shuffle=True)

model = ControlsCNN(num_classes=len(ds.class_names))
trained_model = train_model(model, dl, epochs=1000, lr=1e-3, save_path="controls_cnn.pt")


C:\Users\emmanuelasare\AppData\Local\Temp\ipykernel_21784\3203778515.py:32: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
C:\Users\emmanuelasare\AppData\Local\Temp\ipykernel_21784\3203778515.py:57: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp):


Epoch 01/1000 | Step 0001/0248 | loss=1.1698 acc=0.2188
Epoch 01/1000 | Step 0020/0248 | loss=1.0368 acc=0.5047
Epoch 01/1000 | Step 0040/0248 | loss=1.0134 acc=0.5305
Epoch 01/1000 | Step 0060/0248 | loss=1.0026 acc=0.5281
Epoch 01/1000 | Step 0080/0248 | loss=0.9889 acc=0.5398
Epoch 01/1000 | Step 0100/0248 | loss=0.9840 acc=0.5437
Epoch 01/1000 | Step 0120/0248 | loss=0.9741 acc=0.5505
Epoch 01/1000 | Step 0140/0248 | loss=0.9698 acc=0.5531
Epoch 01/1000 | Step 0160/0248 | loss=0.9607 acc=0.5580
Epoch 01/1000 | Step 0180/0248 | loss=0.9567 acc=0.5590
Epoch 01/1000 | Step 0200/0248 | loss=0.9466 acc=0.5623
Epoch 01/1000 | Step 0220/0248 | loss=0.9374 acc=0.5639
Epoch 01/1000 | Step 0240/0248 | loss=0.9279 acc=0.5677
Epoch 01/1000 | Step 0248/0248 | loss=0.9245 acc=0.5680
Epoch 01 done | loss=0.9245 acc=0.5680 | seen=7931/7931 | time=87.9s
Epoch 02/1000 | Step 0001/0248 | loss=0.7611 acc=0.6562
Epoch 02/1000 | Step 0020/0248 | loss=0.7461 acc=0.6516
Epoch 02/1000 | Step 0040/0248 | lo

KeyboardInterrupt: 

In [ ]:
trained_model = train_model(model, dl, epochs=1000, lr=1e-3, save_path="controls_cnn.pt")

In [10]:
import torch
from PIL import Image
from torchvision import transforms


def predict(
    model,
    image_path: str,
    class_names,
    image_size=(224, 224),
    device: str | None = None,
):
    """
    Predict control class for a single image.

    Returns:
        class_name (str)
        class_index (int)
        confidence (float in [0, 1])
    """

    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"

    model = model.to(device)
    model.eval()

    transform = transforms.Compose([
        transforms.Resize(image_size),
        transforms.ToTensor(),  # divides by 255 -> [0,1]
    ])

    # Load image
    img = Image.open(image_path).convert("RGB")
    x = transform(img).unsqueeze(0).to(device)  # (1, 3, H, W)

    with torch.no_grad():
        logits = model(x)
        probs = torch.softmax(logits, dim=1)
        conf, pred_idx = probs.max(dim=1)

    class_index = pred_idx.item()
    confidence = conf.item()
    class_name = class_names[class_index]

    return class_name, class_index, confidence


In [11]:
# class names come from the dataset
class_names = ds.class_names  # ['Left', 'None', 'Right']

img_path = r"D:\Data\Game Dataset\Archive\az_recorder_20260103_125453s_224x101_Right_cropped\frame_077341_t_001318017807us.jpg"

label, idx, conf = predict(
    model=model,
    image_path=img_path,
    class_names=class_names,
    image_size=(224, 224)
)

print(f"Prediction: {label} (idx={idx}, conf={conf:.3f})")


Prediction: Right (idx=2, conf=1.000)


In [12]:
import csv
from pathlib import Path
from PIL import Image
import torch
from torchvision import transforms


def predict_controls_to_csv(
    model,
    images_dir: str,
    out_csv_path: str,
    class_names,
    left_roi=(0, 32, 91, 100),
    right_roi=(132, 32, 223, 100),
    image_size=(224, 224),
    device: str | None = None,
):
    """
    Predict left/right controls from sequential frame images
    and write CSV with columns: filename,left,right
    """

    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"

    model = model.to(device)
    model.eval()

    transform = transforms.Compose([
        transforms.Resize(image_size),
        transforms.ToTensor(),  # /255
    ])

    images_dir = Path(images_dir)
    out_csv_path = Path(out_csv_path)
    out_csv_path.parent.mkdir(parents=True, exist_ok=True)

    valid_exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

    # 🔑 Explicit sorting to preserve frame order
    image_paths = sorted(
        [p for p in images_dir.iterdir() if p.suffix.lower() in valid_exts],
        key=lambda p: p.name
    )

    idx_to_name = {i: n for i, n in enumerate(class_names)}

    def clamp_box(box, w, h):
        xmin, ymin, xmax, ymax = box
        return (
            max(0, min(xmin, w)),
            max(0, min(ymin, h)),
            max(0, min(xmax, w)),
            max(0, min(ymax, h)),
        )

    def predict_crop(pil_crop):
        x = transform(pil_crop).unsqueeze(0).to(device)
        with torch.no_grad():
            logits = model(x)
            pred_idx = int(torch.argmax(logits, dim=1))
        return idx_to_name[pred_idx]

    rows = []

    for img_path in image_paths:
        with Image.open(img_path) as im:
            im = im.convert("RGB")
            w, h = im.size

            left_crop = im.crop(clamp_box(left_roi, w, h))
            right_crop = im.crop(clamp_box(right_roi, w, h))

        left_pred = predict_crop(left_crop)
        right_pred = predict_crop(right_crop)

        left_val = 1 if left_pred == "Left" else 0
        right_val = 1 if right_pred == "Right" else 0

        rows.append([img_path.name, left_val, right_val])

    with open(out_csv_path, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["filename", "left", "right"])
        writer.writerows(rows)

    print(f"Wrote {len(rows)} rows to {out_csv_path}")


In [13]:
predict_controls_to_csv(
    model=model,
    images_dir=r"D:\Data\Game Dataset\az_recorder_20251231_165942_GamePlay_224x224",
    out_csv_path=r"D:\Data\Game Dataset\az_recorder_20251231_165942_GamePlay_224x224_GamePlay_224x224_controls_extracted.csv",
    class_names=ds.class_names,  # ['Left','None','Right']
    left_roi=(0, 102, 91, 148),
    right_roi=(132, 102, 223, 148),
    image_size=(224, 224),
    device="cuda",
)

Wrote 238023 rows to D:\Data\Game Dataset\az_recorder_20251231_165942_GamePlay_224x224_GamePlay_224x224_controls_extracted.csv


In [ ]:
predict_controls_to_csv(
    model=model,
    images_dir=r"D:\Data\Game Dataset\az_recorder_20260103_213242_GamePlay_224x224",
    out_csv_path=r"D:\Data\Game Dataset\az_recorder_20260103_213242_GamePlay_224x224_controls_extracted.csv",
    class_names=ds.class_names,  # ['Left','None','Right']
    left_roi=(0, 32, 91, 100),
    right_roi=(132, 32, 223, 100),
    image_size=(224, 224),
)


Wrote 151485 rows to D:\Data\Game Dataset\az_recorder_20260103_125453s_224x101_controls_extracted.csv


In [32]:
import shutil
from pathlib import Path


def detect_class_in_folder(
    model,
    src_dir: str,
    dst_dir: str,
    target_class: str,
    class_names,
    image_size=(224, 224),
    device: str | None = None,
    min_confidence: float = 0.0,
):
    """
    Run prediction on all images in src_dir and copy those predicted
    as target_class into dst_dir.

    Args:
        model: trained model
        src_dir: folder with images
        dst_dir: destination folder
        target_class: class name to detect (e.g. 'Right')
        class_names: list of class names
        image_size: resize used during training
        min_confidence: optional confidence threshold
    """

    src_dir = Path(src_dir)
    dst_dir = Path(dst_dir)
    dst_dir.mkdir(parents=True, exist_ok=True)

    valid_exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

    images = sorted([p for p in src_dir.iterdir() if p.suffix.lower() in valid_exts])
    if not images:
        print("No images found.")
        return

    kept = 0

    for img_path in images:
        label, idx, conf = predict(
            model=model,
            image_path=str(img_path),
            class_names=class_names,
            image_size=image_size,
            device=device,
        )

        if label == target_class and conf >= min_confidence:
            shutil.copy2(img_path, dst_dir / img_path.name)
            kept += 1

    print(
        f"Detected '{target_class}' in {kept}/{len(images)} images "
        f"(min_confidence={min_confidence})"
    )


In [41]:
detect_class_in_folder(
    model=model,
    src_dir=r"D:\Data\Game Dataset\az_recorder_20260103_125453s_224x101_Left_cropped",
    dst_dir=r"D:\Data\Game Dataset\Detected_Left",
    target_class="Left",
    class_names=ds.class_names,
    image_size=(224, 224),
    min_confidence=0.7,
)


Detected 'Left' in 17475/151485 images (min_confidence=0.7)


In [ ]:
detect_class_in_folder(
    model=model,
    src_dir=r"D:\Data\Game Dataset\az_recorder_20260103_125453s_224x101_Right_cropped",
    dst_dir=r"D:\Data\Game Dataset\Detected_Right",
    target_class="Right",
    class_names=ds.class_names,
    image_size=(224, 224),
    min_confidence=0.7,
)


Detected 'None' in 134036/151485 images (min_confidence=0.7)


In [3]:
# Resize images from source to destination (224 x 101)
# - Recursively copies images from the source folder into the destination folder
# - Resizes each image to the exact target size (may change aspect ratio)
# - Skips files already present in the destination with the same size

from pathlib import Path
from PIL import Image

# --- Configuration (edit if needed) ---
SRC = Path(r"D:\Data\Game Dataset\ControlsOld\None")
DST = Path(r"D:\Data\Game Dataset\ControlsOld\None_224x101")
TARGET_SIZE = (224, 101)  # (width, height) in pixels
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
VERIFY_SAMPLE = 5  # number of files to show as sample
DRY_RUN = False  # set True to only show what would be done

if not SRC.exists():
    raise FileNotFoundError(f"Source folder not found: {SRC}")

DST.mkdir(parents=True, exist_ok=True)

processed = 0
skipped = 0
errors = 0

all_images = [p for p in SRC.rglob("*") if p.is_file() and p.suffix.lower() in IMG_EXTS]
print(f"Found {len(all_images)} images under: {SRC}")

for i, p in enumerate(all_images, start=1):
    try:
        rel = p.relative_to(SRC)
        out_path = DST / rel
        out_path.parent.mkdir(parents=True, exist_ok=True)

        # If destination exists and has expected size, skip
        if out_path.exists():
            try:
                with Image.open(out_path) as im_out:
                    if im_out.size == (TARGET_SIZE[0], TARGET_SIZE[1]):
                        skipped += 1
                        continue
            except Exception:
                # if opening existing file fails, overwrite
                pass

        if DRY_RUN:
            print(f"[DRY] Would resize: {p} -> {out_path}")
            processed += 1
            continue

        with Image.open(p) as im:
            im = im.convert("RGB")
            # PIL expects size as (width, height)
            im_resized = im.resize(TARGET_SIZE, Image.LANCZOS)
            # Save with same format as original extension (use JPEG for .jpg/.jpeg)
            fmt = "JPEG" if out_path.suffix.lower() in {".jpg", ".jpeg"} else None
            im_resized.save(out_path, format=fmt)
            processed += 1

        if i % 200 == 0:
            print(f"Processed {i}/{len(all_images)}")

    except Exception as e:
        errors += 1
        print(f"Error processing {p}: {e}")

print("--- Done ---")
print(f"Processed: {processed}")
print(f"Skipped (already present with same size): {skipped}")
print(f"Errors: {errors}")

# Quick verification sample (show sizes for a few files)
from itertools import islice
print('\nSample of resized images (path -> size):')
for p in islice(sorted(DST.rglob("*")), VERIFY_SAMPLE):
    if p.is_file() and p.suffix.lower() in IMG_EXTS:
        try:
            with Image.open(p) as im:
                print(p.relative_to(DST), '->', im.size)
        except Exception:
            print(p.relative_to(DST), '-> failed to open')


Found 22 images under: D:\Data\Game Dataset\ControlsOld\None
--- Done ---
Processed: 22
Skipped (already present with same size): 0
Errors: 0

Sample of resized images (path -> size):
frame_000006_t_000000102249us.jpg -> (224, 101)
frame_013118_t_000223552289us.jpg -> (224, 101)
frame_026720_t_000455352734us.jpg -> (224, 101)
frame_035553_t_000605881577us.jpg -> (224, 101)
frame_038697_t_000659460507us.jpg -> (224, 101)
--- Done ---
Processed: 22
Skipped (already present with same size): 0
Errors: 0

Sample of resized images (path -> size):
frame_000006_t_000000102249us.jpg -> (224, 101)
frame_013118_t_000223552289us.jpg -> (224, 101)
frame_026720_t_000455352734us.jpg -> (224, 101)
frame_035553_t_000605881577us.jpg -> (224, 101)
frame_038697_t_000659460507us.jpg -> (224, 101)


In [15]:
import os
from pathlib import Path
from PIL import Image


def crop_images_in_folder(
    src_dir: str,
    dst_dir: str,
    xmin: int,
    ymin: int,
    xmax: int,
    ymax: int,
):
    """
    Crop all images in src_dir using (xmin, ymin, xmax, ymax)
    and save them to dst_dir with the same filenames.
    """

    src_dir = Path(src_dir)
    dst_dir = Path(dst_dir)
    dst_dir.mkdir(parents=True, exist_ok=True)

    valid_exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

    images = sorted([p for p in src_dir.iterdir() if p.suffix.lower() in valid_exts])

    if not images:
        print("No images found.")
        return

    for img_path in images:
        try:
            with Image.open(img_path) as img:
                img = img.convert("RGB")

                w, h = img.size

                # Clamp crop box to image bounds
                x1 = max(0, xmin)
                y1 = max(0, ymin)
                x2 = min(w, xmax)
                y2 = min(h, ymax)

                if x2 <= x1 or y2 <= y1:
                    print(f"Skipping {img_path.name}: invalid crop box")
                    continue

                cropped = img.crop((x1, y1, x2, y2))
                cropped.save(dst_dir / img_path.name)

        except Exception as e:
            print(f"Failed on {img_path.name}: {e}")

    print(f"Cropped {len(images)} images into {dst_dir}")


In [25]:
src_folder = r"D:\Data\Game Dataset\az_recorder_20260103_125453s_224x101"
dst_folder = r"D:\Data\Game Dataset\az_recorder_20260103_125453s_224x101_Left_cropped"

# Example ROI
# Left Roi
xmin, ymin, xmax, ymax = 0, 32, 91, 100

crop_images_in_folder(
    src_dir=src_folder,
    dst_dir=dst_folder,
    xmin=xmin,
    ymin=ymin,
    xmax=xmax,
    ymax=ymax,
)


Cropped 151485 images into D:\Data\Game Dataset\az_recorder_20260103_125453s_224x101_Left_cropped


In [29]:
src_folder = r"D:\Data\Game Dataset\az_recorder_20260103_125453s_224x101"
dst_folder = r"D:\Data\Game Dataset\Controls\az_recorder_20260103_125453s_224x101_Right_cropped"

# Example ROI
xmin, ymin, xmax, ymax = 132, 32, 223, 100

crop_images_in_folder(
    src_dir=src_folder,
    dst_dir=dst_folder,
    xmin=xmin,
    ymin=ymin,
    xmax=xmax,
    ymax=ymax,
)

Cropped 151485 images into D:\Data\Game Dataset\Controls\az_recorder_20260103_125453s_224x101_Right_cropped
